# 01 — Coleta e preparação

Primeira etapa do Projeto 01. Este notebook baixa os arquivos do SIH-SUS, valida o que veio e
constrói a **tabela analítica** que a análise exploratória vai consumir.

A ordem importa: nada de gráfico antes de saber que o dado está íntegro. Cada checagem aqui
existe porque uma das advertências de domínio do [README](../README.md) poderia invalidar a
análise se passasse em branco.

## Preparação

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().parents[2]
sys.path.insert(0, str(Path.cwd().parent / "src"))

import coleta
import dicionario as dic

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 40)

UF, ANO = "SC", 2024
DADOS = RAIZ / "dados"
print(f"escopo: {UF} / {ANO}")
print(f"dados brutos em: {DADOS / 'bruto' / 'sihsus'}")

## 1. Coleta

O download é idempotente — rodar de novo não baixa o que já está em disco. Para Santa Catarina
são 12 arquivos de cerca de 3,7 MB, então a primeira execução leva alguns minutos.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(message)s", datefmt="%H:%M:%S")

consolidado = coleta.coletar(UF, ANO, meses=list(range(1, 13)))
consolidado

## 2. Validação do dado bruto

In [ ]:
bruto = coleta.validar(consolidado)

## 3. Tabela analítica

`dicionario.rotular()` traduz os códigos para rótulos legíveis e deriva as colunas que a análise
usa: idade em anos (tratando o código 5, que significa "acima de 100"), faixa de idade, capítulo
do CID-10, uso de UTI e atendimento fora do município de residência.

In [ ]:
analitico = dic.rotular(bruto)

derivadas = [c for c in analitico.columns if c not in bruto.columns]
print("colunas derivadas:", derivadas)
analitico[["idade_anos", "faixa_idade", "sexo", "carater", "complexidade",
           "cid_grupo", "usou_uti", "fora_do_municipio", "DIAS_PERM", "VAL_TOT"]].head()

## 4. Checagens de qualidade

Cinco verificações. Nenhuma é opcional: cada uma corresponde a uma armadilha documentada, e o
resultado de todas vai para a seção de limitações do relatório.

### 4.1 A idade foi convertida corretamente?

In [ ]:
# O teste decisivo: para COD_IDADE = 5, a idade derivada tem que bater com a data de nascimento.
amostra = analitico.loc[analitico["COD_IDADE"] == "5",
                        ["IDADE", "COD_IDADE", "NASC", "DT_INTER", "idade_anos"]].head(8).copy()
amostra["idade_pela_data"] = ((amostra["DT_INTER"] - amostra["NASC"]).dt.days / 365.25).round(0)
print(amostra.to_string(index=False))
print(f"\nidade máxima na base: {analitico['idade_anos'].max():.0f} anos")
print(f"registros sem idade  : {analitico['idade_anos'].isna().sum()}")

### 4.2 `DIAS_PERM` é consistente com as datas?

In [ ]:
dias_pelas_datas = (analitico["DT_SAIDA"] - analitico["DT_INTER"]).dt.days
divergencia = (dias_pelas_datas - analitico["DIAS_PERM"]).abs()

print(f"registros com divergência: {(divergencia > 0).sum():,}".replace(",", ".")
      + f"  ({(divergencia > 0).mean():.2%})")
print(f"divergência máxima       : {divergencia.max()} dias")
print("\ndistribuição da divergência:")
print(divergencia.value_counts().head(5).to_string())

### 4.3 Onde estão as bordas de competência?

Os arquivos agrupam por competência de **pagamento**. Se houver internações muito antigas, os
primeiros e últimos meses da série por `DT_INTER` estarão incompletos e não podem ser comparados
com os meses do meio.

In [ ]:
por_mes = analitico["DT_INTER"].dt.to_period("M").value_counts().sort_index()
print(por_mes.to_string())
print(f"\nperíodo completo: {por_mes.index.min()} a {por_mes.index.max()}")
print(f"internações fora de {ANO}: {(analitico['DT_INTER'].dt.year != ANO).sum():,}".replace(",", "."))

### 4.4 Existem AIH de continuação, que duplicariam a permanência?

In [ ]:
print(analitico["tipo_aih"].value_counts(dropna=False).to_string())
print(f"\nAIH duplicadas (mesmo N_AIH): {analitico['N_AIH'].duplicated().sum():,}".replace(",", "."))

### 4.5 Há valores impossíveis?

In [ ]:
suspeitas = {
    "DIAS_PERM negativo":        (analitico["DIAS_PERM"] < 0).sum(),
    "VAL_TOT <= 0":              (analitico["VAL_TOT"] <= 0).sum(),
    "idade acima de 120 anos":   (analitico["idade_anos"] > 120).sum(),
    "saída antes da internação": (analitico["DT_SAIDA"] < analitico["DT_INTER"]).sum(),
    "UTI sem valor de UTI":      ((analitico["UTI_MES_TO"] > 0) & (analitico["VAL_UTI"] == 0)).sum(),
}
for rotulo, n_casos in suspeitas.items():
    print(f"  {rotulo:<28} {n_casos:>8,}".replace(",", ".") + f"   ({n_casos/len(analitico):.3%})")

## 5. Gravando a tabela analítica

O Parquet analítico é o insumo do notebook 02. A amostra em `dados/exemplo/` é pequena o
suficiente para ir para o Git, o que deixa o repositório navegável por quem não vai baixar
600 mil linhas do DATASUS.

In [ ]:
destino = DADOS / "bruto" / "sihsus" / f"analitico-{UF}-{ANO}.parquet"
analitico.to_parquet(destino, index=False)
print(f"{destino.name}: {destino.stat().st_size / 1024**2:.1f} MB, {len(analitico):,} linhas".replace(",", "."))

amostra = analitico.sample(min(5_000, len(analitico)), random_state=42)
exemplo = DADOS / "exemplo" / f"sihsus-{UF}-{ANO}-amostra.parquet"
amostra.to_parquet(exemplo, index=False)
print(f"{exemplo.name}: {exemplo.stat().st_size / 1024:.0f} KB, {len(amostra):,} linhas".replace(",", "."))

## Fechamento

O que registrar antes de seguir para o notebook 02:

- Quantas internações sobraram, e de que período efetivo.
- Quais checagens acusaram problema, com o percentual afetado.
- Que decisão você tomou para cada problema — descartar, corrigir ou manter com advertência.

Essa terceira linha é a que mais importa. Uma decisão registrada é análise; a mesma decisão
tomada em silêncio é um número que ninguém pode auditar.